# Train people-counter-detector (Kaggle)

Re-train de yolov8n sobre el **dataset YOLO local** generado por
`labelme_to_yolo.py` (post-labeling en X-AnyLabeling) y subido a
Kaggle como **dataset privado**. Único template para iteraciones del
modelo — al cambiar de versión, solo se actualiza el slug del dataset
attached + el `name` del run.

- yolov8n único (validado contra 8s/11n/11s en el primer comparativo — gana
  por margen de Hailo-8L)
- Hiperparams baseline: epochs=100, imgsz=640, batch=16, patience=20
- Eval en val set held-out + export ONNX. ~20 min en T4 x2.

**Antes de Save & Run All**:
1. Subir / versionar el dataset privado a Kaggle (`kaggle datasets version -p training_data/dataset_v_next -m "v_next batch NN"`).
2. Attach el dataset al notebook (sidebar derecho → Add Data → tu dataset privado).
3. Sufijá el `name` del run en Cell 4 (ej. `people-counter-detector-v3`) para diferenciar runs.

**Histórico de iteraciones** (eval contra val held-out 245 imgs / 174 cajas — `scripts/training/README.md` tiene la tabla completa con precision/recall):
- v1: primera iter X-AnyLabeling — 294 imgs / 133 cajas, mAP50 **0.805**, mAP50-95 0.385.
- **v2: + active learning (250 imgs) — 544 imgs / 438 cajas, mAP50 0.956, mAP50-95 0.567, 26.8 FPS — deployado producción 2026-05-20.**
- v3: 2da iter AL (+250 imgs) — mAP50 0.939, mAP50-95 0.538 → descartado, peor que v2 en todo excepto recall.

In [ ]:
!pip install -q ultralytics

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import glob
import os
import shutil
import yaml

# Localizar el data.yaml del dataset privado attacheado.
# Kaggle monta los datasets en /kaggle/input/<slug>/ pero la API a veces
# anida bajo /kaggle/input/datasets/<user>/<slug>/ (ver memoria
# kaggle_automation_via_api). El glob amplio cubre ambos casos.
candidates = sorted(glob.glob("/kaggle/input/**/data.yaml", recursive=True))
if not candidates:
    raise FileNotFoundError(
        "No se encontró data.yaml en /kaggle/input/**. "
        "Attach el dataset privado al notebook (sidebar → Add Data)."
    )
src_yaml = candidates[0]
src_root = os.path.dirname(src_yaml)
print(f"Dataset encontrado en: {src_root}")

# Copiar a /kaggle/working/dataset/ (writable) para que Ultralytics pueda
# crear su cache de labels al lado de las imgs sin tocar el read-only mount.
dst_root = "/kaggle/working/dataset"
if os.path.exists(dst_root):
    shutil.rmtree(dst_root)
shutil.copytree(src_root, dst_root)

# Re-escribir data.yaml con paths absolutos al working dir (algunos
# data.yaml exportados de labelme_to_yolo.py vienen con paths relativos).
with open(f"{dst_root}/data.yaml") as f:
    data = yaml.safe_load(f) or {}
data = {
    "path": dst_root,
    "train": data.get("train", "train/images"),
    "val": data.get("val", "val/images"),
    # test es opcional — si no está el split, se usa val para eval final.
    **({"test": data["test"]} if data.get("test") else {}),
    "nc": data.get("nc", 1),
    "names": data.get("names", ["person"]),
}
with open(f"{dst_root}/data.yaml", "w") as f:
    yaml.safe_dump(data, f)

# Sanity check del balance (positivos vs negativos por split).
# Un .txt vacío = background revisado (convención de labelme_to_yolo.py).
for split_key in ("train", "val", "test"):
    split_rel = data.get(split_key)
    if not split_rel:
        continue
    img_dir = os.path.join(dst_root, split_rel)
    lbl_dir = img_dir.replace("/images", "/labels")
    if not os.path.isdir(img_dir):
        print(f"  {split_key}: (sin split)"); continue
    n_imgs = len([f for f in os.listdir(img_dir) if not f.startswith(".")])
    if os.path.isdir(lbl_dir):
        lbls = [f for f in os.listdir(lbl_dir) if f.endswith(".txt")]
        n_empty = sum(
            1 for f in lbls
            if os.path.getsize(os.path.join(lbl_dir, f)) == 0
        )
        n_with_bbox = len(lbls) - n_empty
    else:
        n_with_bbox = n_empty = 0
    print(f"  {split_key}: {n_imgs} imgs ({n_with_bbox} con bbox / {n_empty} background revisado)")

In [ ]:
from ultralytics import YOLO

# Cambiar por cada iteración (v4, v5, ...) para diferenciar runs en disk.
name = "people-counter-detector"
print(f"Training {name}")
print("=" * 60)

model = YOLO("yolov8n.pt")
model.train(
    data="/kaggle/working/dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    project="/kaggle/working/runs",
    name=name,
    save=True,
    plots=True,
)
print(f"\n-> /kaggle/working/runs/{name}/weights/best.pt")

In [ ]:
from ultralytics import YOLO
import shutil
import json

src = f"/kaggle/working/runs/{name}/weights"

# Copy PT al working dir para download fácil del panel Output.
shutil.copy(f"{src}/best.pt", f"/kaggle/working/{name}.pt")

# Eval en val held-out (o test si está, ultralytics elige automáticamente).
m = YOLO(f"{src}/best.pt")
eval_split = "test" if data.get("test") else "val"
metrics = m.val(split=eval_split, verbose=False)
results = {
    "name": name,
    "ckpt": "yolov8n.pt",
    "eval_split": eval_split,
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "P": float(metrics.box.mp),
    "R": float(metrics.box.mr),
}

# Export ONNX para HEF compile después.
m.export(format="onnx", imgsz=640, opset=11, simplify=True)
shutil.copy(f"{src}/best.onnx", f"/kaggle/working/{name}.onnx")

print(f"\n{'='*60}\nMÉTRICAS — held-out {eval_split} set\n{'='*60}")
print(f"  mAP@50    : {results['mAP50']:.3f}")
print(f"  mAP@50-95 : {results['mAP50_95']:.3f}")
print(f"  Precision : {results['P']:.3f}")
print(f"  Recall    : {results['R']:.3f}")

with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nDownloads disponibles en panel Output:")
print(f"  {name}.pt + {name}.onnx + results.json")